# Figure 2 — Effect of Varying θ

Reproduces **Figure 2** from *Density-Reweighted Entropic Optimal Transport*.

Uses the same two-manifold setup as Figure 1 (line vs. quadratic curve with opposing sampling densities), but sweeps θ ∈ {0, 1/3, 2/3, 1} to show how increasing θ transitions from density-driven (θ=0, same as standard EOT) to geometry-driven (θ=1) correspondences.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from sklearn.metrics import pairwise_distances
import matplotlib as mpl
import matplotlib.pyplot as plt

from dreot import sinkhorn_dreot

## 1. Data Generation  (same as Figure 1)

In [ ]:
class PiecewiseUniform:
    def __init__(self, break_point=0.5, weights=(1, 1)):
        self.bp = break_point
        prob = break_point * weights[0] + (1 - break_point) * weights[1]
        self.w = [weights[0] / prob, weights[1] / prob]
        self.coin = break_point * weights[0] / prob

    def sample(self, n):
        c = np.random.uniform(size=n)
        n1 = int(np.sum(c <= self.coin))
        x = np.concatenate([
            np.random.uniform(0, self.bp, n1),
            np.random.uniform(self.bp, 1, n - n1),
        ])
        return np.sort(x).reshape(-1, 1)

    def pdf(self, x):
        return np.where(np.asarray(x) < self.bp, self.w[0], self.w[1])


np.random.seed(42)
m, n    = 2000, 3000
c_curve = 0.5
eps     = 5e-2

samplerX = PiecewiseUniform(0.5, (9, 1))
samplerY = PiecewiseUniform(0.5, (1, 9))

x_X = samplerX.sample(m)
x_Y = samplerY.sample(n)

X = np.hstack([x_X, x_X])
Y = np.hstack([x_Y, 2 + x_Y + c_curve * x_Y**2])

mu = samplerX.pdf(x_X) / np.sqrt(2)
nu = samplerY.pdf(x_Y) / np.sqrt(1 + (1 + 2*c_curve*x_Y)**2)

dist = pairwise_distances(X, Y, metric="sqeuclidean")
print("Data ready.")

## 2. Compute DR-EOT for θ ∈ {0, 1/3, 2/3, 1}

In [ ]:
theta_vals = np.linspace(0, 1, 4)
W_dict = {}

for theta in theta_vals:
    row_s, col_s = sinkhorn_dreot(
        dist, eps,
        mu.reshape(-1, 1), nu.reshape(-1, 1),
        alpha=theta,
        delta=1e-6, max_iter=3000, check_freq=100,
        raise_on_bad_convergence=False,
    )
    W_dict[theta] = row_s * np.exp(-dist / eps) * col_s.T
    print(f"θ = {theta:.2f}  done")

## 3. Figure 2 — Vertical four-panel plot

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Termes", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 9, "axes.titlesize": 9,
    "axes.linewidth": 0.6, "axes.grid": False,
    "figure.dpi": 150, "savefig.dpi": 600,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

ARROW_COLOR = "gray"
vmin = min(mu.min(), nu.min())
vmax = max(mu.max(), nu.max())

titles = [rf"$\theta = {t:.2f}$" for t in theta_vals]

fig, axes = plt.subplots(4, 1, figsize=(2, 3), sharey=False)
fig.subplots_adjust(bottom=0.06, left=0.15, hspace=0.15)

for ax, theta, title in zip(axes, theta_vals, titles):
    sc = ax.scatter(X[:, 0], X[:, 1], c=mu, cmap="viridis",
                    s=5, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True, zorder=3)
    ax.scatter(Y[:, 0], Y[:, 1], c=nu, cmap="viridis",
               s=5, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True, zorder=3)

    max_idx = np.argmax(W_dict[theta], axis=1)
    step = max(1, m // 100)
    for i in range(0, m, step):
        j = max_idx[i]
        ax.annotate("",
                    xy=(Y[j, 0], Y[j, 1]),
                    xytext=(X[i, 0], X[i, 1]),
                    arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR,
                                   lw=0.5, alpha=0.4, mutation_scale=4),
                    zorder=2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(-0.06, 0.5, title, transform=ax.transAxes,
            fontsize=6, rotation=90, va="center", ha="center")

cax = fig.add_axes([0.15, 0.02, 0.70, 0.018])
cbar = fig.colorbar(sc, cax=cax, orientation="horizontal")
cbar.set_label("Sampling Density", fontsize=6, labelpad=3)
cbar.set_ticks([])

plt.savefig("fig2_theta_sweep.pdf", bbox_inches="tight", dpi=600)
plt.show()
print("Saved fig2_theta_sweep.pdf")